# Veteran Performance Features

Build per-season/team veteran performance summary features from supabase export data.

**Sources:**
- `data/supabase_exports/av.csv` — per-player Approximate Value (AV) by season/team, with experience
- `data/supabase_exports/rosters.csv` — roster data with age, used to enrich AV rows where pfr_id links exist

**Veteran definition:** players with `experience > 0` (excludes anyone in their rookie season).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../../data/supabase_exports')
OUTPUT_DIR = Path('../../data/roster')

## Load Data

In [2]:
av = pd.read_csv(DATA_DIR / 'av.csv')
rosters = pd.read_csv(DATA_DIR / 'rosters.csv')

print('av shape:', av.shape)
print('rosters shape:', rosters.shape)

av shape: (55014, 12)
rosters shape: (60308, 38)


In [3]:
av.head(3)

,av_row_id,season,team,team_raw,franchise_id,pfr_player_id,player_name,position,position_group,experience,experience_raw,av
0,1999_CLE_AbduKa00,1999,CLE,CLE,CLE,AbduKa00,Karim Abdul-Jabbar,RB,RB,3,3,3.0
1,1999_CLE_AbduRa20,1999,CLE,CLE,CLE,AbduRa20,Rahim Abdullah,LLB,OTHER,0,Rook,4.0
2,1999_CLE_AlexDe20,1999,CLE,CLE,CLE,AlexDe20,Derrick Alexander,RDE,OTHER,4,4,5.0


In [4]:
rosters[['pfr_id','season','team','age','is_rookie','position_group','roster_position']].head(3)

,pfr_id,season,team,age,is_rookie,position_group,roster_position
0,NaN,2002,ARZ,22.0,0,DB,SS
1,NaN,2002,ARZ,26.0,0,OL,T
2,NaN,2002,ARZ,25.0,0,WR,WR


## Quick Exploration

In [5]:
print('AV experience_raw value counts (top 5):')
print(av['experience_raw'].value_counts().head(5))
print()
print('AV position_group unique:', av['position_group'].unique())
print()
print('ST breakdown by position:')
print(av[av['position_group'] == 'ST']['position'].value_counts())

AV experience_raw value counts (top 5):
experience_raw
Rook    9728
1       9286
2       7124
3       6239
4       5132
Name: count, dtype: int64

AV position_group unique: ['RB' 'OTHER' 'DL' 'OL' 'TE' 'WR' 'QB' 'ST' 'DB' 'LB']

ST breakdown by position:
position
K     1191
P     1023
LS     750
KR       3
PR       3
Name: count, dtype: int64


In [6]:
print('Season range — av:', av['season'].min(), '-', av['season'].max())
print('Season range — rosters:', rosters['season'].min(), '-', rosters['season'].max())
print()
print('Rosters with pfr_id:', rosters['pfr_id'].notna().sum(), '/', len(rosters))

Season range — av: 1999 - 2025
Season range — rosters: 2002 - 2025

Rosters with pfr_id: 19925 / 60308


---
# Section: Veteran Performance Summary

For each `(season, team, pos_category)` combination, summarise veteran performance:

| Position group | Feature logic |
|---|---|
| **QB** | Highest AV + that player's age |
| **K** | Highest AV + that player's age |
| **P** | Highest AV + that player's age |
| **All others** (OL, WR, DL, RB, TE, LB, DB, ST-other, OTHER) | Mean of top-3 AVs + mean of top-3 ages |

Output is **long format**: one row per `(season, team, pos_category)`.  
Columns: `season`, `team`, `pos_category`, `vet_av`, `vet_age`.

Age is included where it can be joined from the rosters table; otherwise `vet_age` is NaN.

**Rookie exclusion:** `experience == 0` in av.csv maps to `experience_raw == 'Rook'`; these rows are dropped before all aggregation.

### Step 1 — Filter veterans and assign position category

In [7]:
# Drop rookies
av_vet = av[av['experience'] > 0].copy()
print('Veteran rows:', len(av_vet), '(dropped', len(av) - len(av_vet), 'rookies)')

Veteran rows: 45284 (dropped 9730 rookies)


In [8]:
# K and P live inside position_group='ST' in av.csv; extract by the `position` column.
# Assign a fine-grained pos_category used for aggregation logic.
def assign_pos_category(row):
    if row['position_group'] == 'QB':
        return 'QB'
    if row['position'] == 'K':
        return 'K'
    if row['position'] == 'P':
        return 'P'
    return row['position_group']  # OL, WR, DL, RB, TE, LB, DB, ST, OTHER

av_vet['pos_category'] = av_vet.apply(assign_pos_category, axis=1)
print(av_vet['pos_category'].value_counts())

pos_category
OTHER    10318
DB        6796
WR        4929
DL        4232
LB        4055
RB        3943
OL        3693
TE        2768
QB        1899
K         1045
P          906
ST         700
Name: count, dtype: int64


### Step 2 — Enrich with age via rosters join

In [9]:
# Join av -> rosters on pfr_player_id == pfr_id, same season and team.
# Only rosters rows with a pfr_id can participate; age coverage will be partial.
roster_age = rosters[rosters['pfr_id'].notna()][['pfr_id', 'season', 'team', 'age']].drop_duplicates()

av_vet = av_vet.merge(
    roster_age,
    left_on=['pfr_player_id', 'season', 'team'],
    right_on=['pfr_id', 'season', 'team'],
    how='left'
).drop(columns='pfr_id')

age_coverage = av_vet['age'].notna().mean()
print(f'Age coverage after join: {age_coverage:.1%} of veteran rows')

Age coverage after join: 17.5% of veteran rows


### Step 3 — Aggregate QB / K / P  (top-1 AV)

In [10]:
SINGLETON_POS = ['QB', 'K', 'P']

singleton_rows = []
for pos in SINGLETON_POS:
    sub = av_vet[av_vet['pos_category'] == pos]
    idx = sub.groupby(['season', 'team'])['av'].idxmax()
    top = sub.loc[idx, ['season', 'team', 'av', 'age']].copy()
    top['pos_category'] = pos
    top = top.rename(columns={'av': 'vet_av', 'age': 'vet_age'})
    singleton_rows.append(top)

singleton_long = pd.concat(singleton_rows, ignore_index=True)
print('Singleton rows:', len(singleton_long))
singleton_long.head(5)

Singleton rows: 2452


,season,team,vet_av,vet_age,pos_category
0,1999,ATL,8.0,NaN,QB
1,1999,BUF,14.0,NaN,QB
2,1999,CAR,19.0,NaN,QB
3,1999,CHI,4.0,NaN,QB
4,1999,CIN,10.0,NaN,QB


### Step 4 — Aggregate all other position groups (mean of top-3 AV)

In [11]:
other_pos_groups = [p for p in av_vet['pos_category'].unique() if p not in SINGLETON_POS]
print('Other position groups:', sorted(other_pos_groups))

Other position groups: ['DB', 'DL', 'LB', 'OL', 'OTHER', 'RB', 'ST', 'TE', 'WR']


In [12]:
def top3_agg(g):
    top3 = g.nlargest(3, 'av')
    vet_av = top3['av'].mean()
    vet_age = top3['age'].mean() if top3['age'].notna().any() else np.nan
    return pd.Series({'vet_av': vet_av, 'vet_age': vet_age})

other_long = (
    av_vet[av_vet['pos_category'].isin(other_pos_groups)]
    .groupby(['season', 'team', 'pos_category'])
    .apply(top3_agg, include_groups=False)
    .reset_index()
)
print('Other-group rows:', len(other_long))
other_long.head(5)

Other-group rows: 7503


,season,team,pos_category,vet_av,vet_age
0,1999,ATL,DB,4.333333,NaN
1,1999,ATL,DL,0.666667,NaN
2,1999,ATL,LB,2.666667,NaN
3,1999,ATL,OL,2.666667,NaN
4,1999,ATL,OTHER,7.000000,NaN


### Step 5 — Combine into a long-format season / team / pos_category table

In [13]:
features = (
    pd.concat([singleton_long, other_long], ignore_index=True)
    .sort_values(['season', 'team', 'pos_category'])
    .reset_index(drop=True)
)

print('Final feature table shape:', features.shape)
features.head(12)

Final feature table shape: (9955, 5)


,season,team,vet_av,vet_age,pos_category
0,1999,ATL,4.333333,NaN,DB
1,1999,ATL,0.666667,NaN,DL
2,1999,ATL,2.000000,NaN,K
3,1999,ATL,2.666667,NaN,LB
4,1999,ATL,2.666667,NaN,OL
5,1999,ATL,7.000000,NaN,OTHER
6,1999,ATL,1.000000,NaN,P
7,1999,ATL,8.000000,NaN,QB
8,1999,ATL,4.000000,NaN,RB
9,1999,ATL,2.000000,NaN,ST


In [14]:
print('Rows per pos_category:')
print(features['pos_category'].value_counts().sort_index())
print()
print('Null rates:')
print((features.isna().mean() * 100).round(1).to_string())

Rows per pos_category:
pos_category
DB       861
DL       860
K        813
LB       861
OL       859
OTHER    834
P        784
QB       855
RB       861
ST       645
TE       861
WR       861
Name: count, dtype: int64

Null rates:
season           0.0
team             0.0
vet_av           0.0
vet_age         76.3
pos_category     0.0


In [15]:
features.describe().round(2)

,season,vet_av,vet_age
count,9955.00,9955.00,2357.00
mean,2012.06,4.59,26.50
std,7.76,3.56,2.71
min,1999.00,-2.00,21.00
25%,2005.00,2.00,24.67
50%,2012.00,3.67,26.00
75%,2019.00,6.00,28.00
max,2025.00,25.00,41.00


### Save

In [16]:
out_path = OUTPUT_DIR / 'veteran_performance_features.csv'
features.to_csv(out_path, index=False)
print(f'Saved {len(features)} rows to {out_path}')

Saved 9955 rows to ../../data/roster/veteran_performance_features.csv


---
# Section: Positional Draft Context Features (Team-Level)

For each draft pick `(draft_season, pick, round, team)`, add two features based on picks made **earlier in the same draft by the same team** (strictly lower overall pick number), within the same `position_group`:

| Column | Logic |
|---|---|
| `pos_drafted` | `1` if this team already picked someone of the same position group earlier in this draft, `0` otherwise |
| `pos_first_overall` | Overall pick number of the team's first such earlier pick; `NaN` if this is the team's first pick of that position group |

A team's first pick of any position group always gets `pos_drafted=0` / `pos_first_overall=NaN`.

In [17]:
drafts = pd.read_csv(DATA_DIR / 'drafts.csv')

draft_picks = drafts[['draft_season', 'round', 'pick', 'team', 'position_group']].copy()
draft_picks = draft_picks.sort_values(['draft_season', 'pick']).reset_index(drop=True)

print('Draft picks shape:', draft_picks.shape)
print('Season range:', draft_picks['draft_season'].min(), '-', draft_picks['draft_season'].max())
draft_picks.head(5)

Draft picks shape: (12670, 5)
Season range: 1980 - 2025


,draft_season,round,pick,team,position_group
0,1980,1,1,DET,RB
1,1980,1,2,NYJ,WR
2,1980,1,3,CIN,OL
3,1980,1,4,GNB,DL
4,1980,1,5,BAL,RB


In [18]:
# Within each (season, team, position_group) group, find the first prior same-position
# pick this team made. shift(1) excludes the current pick; expanding().min() carries
# the earliest overall seen so far.
draft_picks['pos_first_overall'] = (
    draft_picks
    .groupby(['draft_season', 'team', 'position_group'])['pick']
    .transform(lambda s: s.shift(1).expanding().min())
)

draft_picks['pos_drafted'] = draft_picks['pos_first_overall'].notna().astype(int)

draft_picks = draft_picks[['draft_season', 'pick', 'round', 'team', 'position_group', 'pos_drafted', 'pos_first_overall']]
draft_picks.head(10)

,draft_season,pick,round,team,position_group,pos_drafted,pos_first_overall
0,1980,1,1,DET,RB,0,NaN
1,1980,2,1,NYJ,WR,0,NaN
2,1980,3,1,CIN,OL,0,NaN
3,1980,4,1,GNB,DL,0,NaN
4,1980,5,1,BAL,RB,0,NaN
5,1980,6,1,STL,DL,0,NaN
6,1980,7,1,ATL,TE,0,NaN
7,1980,8,1,NYG,DB,0,NaN
8,1980,9,1,MIN,DL,0,NaN
9,1980,10,1,SEA,DL,0,NaN


In [19]:
print('pos_drafted value counts:')
print(draft_picks['pos_drafted'].value_counts())
print()

# A team's very first pick of any position group should always be pos_drafted=0
print('Null rate of pos_first_overall when pos_drafted==0:',
      draft_picks[draft_picks['pos_drafted']==0]['pos_first_overall'].isna().mean())
print('Null rate of pos_first_overall when pos_drafted==1:',
      draft_picks[draft_picks['pos_drafted']==1]['pos_first_overall'].isna().mean())

pos_drafted value counts:
pos_drafted
0    8108
1    4562
Name: count, dtype: int64

Null rate of pos_first_overall when pos_drafted==0: 1.0
Null rate of pos_first_overall when pos_drafted==1: 0.0


In [20]:
# Find a team in 2000 that drafted multiple QBs to verify the logic
multi_qb_teams = (
    draft_picks[(draft_picks['draft_season']==2000) & (draft_picks['position_group']=='QB')]
    .groupby('team').filter(lambda g: len(g) > 1)['team'].unique()
)
print('2000 teams with 2+ QB picks:', multi_qb_teams)

if len(multi_qb_teams):
    t = multi_qb_teams[0]
    sample = draft_picks[
        (draft_picks['draft_season']==2000) &
        (draft_picks['team']==t) &
        (draft_picks['position_group']=='QB')
    ]
    print(sample[['draft_season','team','pick','round','pos_drafted','pos_first_overall']])

2000 teams with 2+ QB picks: ['SFO']
      draft_season team  pick  round  pos_drafted  pos_first_overall
6090          2000  SFO    65      3            0                NaN
6237          2000  SFO   212      7            1               65.0


### Save

In [21]:
out_path = OUTPUT_DIR / 'draft_positional_context_features.csv'
draft_picks.to_csv(out_path, index=False)
print(f'Saved {len(draft_picks)} rows to {out_path}')

Saved 12670 rows to ../../data/roster/draft_positional_context_features.csv


---
# Section: Pick Trade Flags

For each draft pick `(season, pick, round, team)`, flag whether the pick was acquired via trade and provide context about that trade.

| Column | Logic |
|---|---|
| `was_traded` | `1` if the pick changed hands in any trade, `0` otherwise |
| `was_primary_pick` | `1` if it was the lowest overall pick number (highest value) in its determining trade |
| `trade_had_players` | `1` if the determining trade included any players |

**Multi-traded picks:** some picks change hands more than once in a chain. The *determining trade* is the last trade in the chain — the one that delivered the pick to the team that ultimately used it. All flags are computed relative to that final trade.

In [22]:
import json

trades = pd.read_csv(DATA_DIR / 'trades.csv')

# Explode assets_json into one row per non-conditional, known draft pick asset
pick_rows = []
for _, row in trades.iterrows():
    assets = json.loads(row['assets_json'])
    for a in assets:
        if a['asset_type'] == 'draft_pick' and not a['conditional'] and a['pick_number'] is not None:
            pick_rows.append({
                'trade_id':    row['trade_id'],
                'trade_date':  row['trade_date'],
                'has_players': row['has_players'],
                'pick_season': int(a['pick_season']),
                'pick_number': int(a['pick_number']),
                'pick_round':  int(a['pick_round']),
                'to_team':     a['to_team'],
            })

traded = pd.DataFrame(pick_rows)
print('Total traded pick assets (non-conditional):', len(traded))
print('Unique (season, pick_number) combos:', traded[['pick_season','pick_number']].drop_duplicates().shape[0])
traded.head(5)

Total traded pick assets (non-conditional): 3190
Unique (season, pick_number) combos: 2365


,trade_id,trade_date,has_players,pick_season,pick_number,pick_round,to_team
0,702,2002-03-08,1,2002,25,1,NO
1,702,2002-03-08,1,2002,125,4,NO
2,702,2002-03-08,1,2003,18,1,NO
3,702,2002-03-08,1,2002,114,4,MIA
4,704,2002-03-11,1,2002,126,4,NE


In [23]:
# For picks traded multiple times, keep only the last trade (highest trade_id).
# That final leg is what delivered the pick to the team that actually used it.
traded = (
    traded
    .sort_values('trade_id')
    .groupby(['pick_season', 'pick_number'], as_index=False)
    .last()
)
print('Unique traded picks after dedup:', len(traded))

# Flag the primary pick within each determining trade:
# lowest pick_number among all picks in that trade = highest value pick.
trade_min = traded.groupby('trade_id')['pick_number'].min().rename('trade_min_pick')
traded = traded.join(trade_min, on='trade_id')
traded['was_primary_pick'] = (traded['pick_number'] == traded['trade_min_pick']).astype(int)
traded = traded.drop(columns='trade_min_pick')

traded.head(5)

Unique traded picks after dedup: 2365


,pick_season,pick_number,trade_id,trade_date,has_players,pick_round,to_team,was_primary_pick
0,2002,6,711,2002-04-20,0,1,KC,1
1,2002,8,711,2002-04-20,0,1,DAL,0
2,2002,14,712,2002-04-20,0,1,NYG,1
3,2002,15,712,2002-04-20,0,1,TEN,0
4,2002,17,714,2002-04-20,0,1,OAK,1


In [24]:
# Start from the full draft picks universe and left-join trade flags
all_picks = drafts[['draft_season', 'pick', 'round', 'team']].copy()
all_picks = all_picks.rename(columns={'draft_season': 'season'})

trade_flags = traded[['pick_season', 'pick_number', 'was_primary_pick', 'has_players']].rename(columns={
    'pick_season':  'season',
    'pick_number':  'pick',
    'has_players':  'trade_had_players',
})

pick_flags = all_picks.merge(trade_flags, on=['season', 'pick'], how='left')
pick_flags['was_traded']        = pick_flags['trade_had_players'].notna().astype(int)
pick_flags['trade_had_players'] = pick_flags['trade_had_players'].fillna(0).astype(int)
pick_flags['was_primary_pick']  = pick_flags['was_primary_pick'].fillna(0).astype(int)

pick_flags = pick_flags[['season', 'pick', 'round', 'team', 'was_traded', 'was_primary_pick', 'trade_had_players']]
print('Final shape:', pick_flags.shape)
pick_flags.head(8)

Final shape: (12670, 7)


,season,pick,round,team,was_traded,was_primary_pick,trade_had_players
0,1980,1,1,DET,0,0,0
1,1980,2,1,NYJ,0,0,0
2,1980,3,1,CIN,0,0,0
3,1980,4,1,GNB,0,0,0
4,1980,5,1,BAL,0,0,0
5,1980,6,1,STL,0,0,0
6,1980,7,1,ATL,0,0,0
7,1980,8,1,NYG,0,0,0


In [25]:
print('was_traded:',       pick_flags['was_traded'].value_counts().to_dict())
print('was_primary_pick:', pick_flags['was_primary_pick'].value_counts().to_dict())
print('trade_had_players:', pick_flags['trade_had_players'].value_counts().to_dict())
print()
# Sanity: was_primary_pick and trade_had_players should only be 1 when was_traded==1
assert pick_flags.loc[pick_flags['was_traded']==0, 'was_primary_pick'].sum() == 0
assert pick_flags.loc[pick_flags['was_traded']==0, 'trade_had_players'].sum() == 0
print('Sanity checks passed.')

was_traded: {0: 10305, 1: 2365}
was_primary_pick: {0: 11570, 1: 1100}
trade_had_players: {0: 11964, 1: 706}

Sanity checks passed.


In [26]:
out_path = OUTPUT_DIR / 'pick_trade_flags.csv'
pick_flags.to_csv(out_path, index=False)
print(f'Saved {len(pick_flags)} rows to {out_path}')

Saved 12670 rows to ../../data/roster/pick_trade_flags.csv
